In [ ]:
from bokeh.plotting import figure, show
from bokeh.io import output_notebook, curdoc, output_file, reset_output
from bokeh.models import AjaxDataSource
from bokeh.models.widgets import Select
from bokeh.models.callbacks import CustomJS
from bokeh.layouts import column, row, layout, widgetbox

In [ ]:
output_file("interactive_graph.html", title="Filtering data using RESTful API")

In [ ]:
initial_state = r'http://gherka.pythonanywhere.com/data/HB=NHS Tree&Sex=Male&Measure=Big Count'
source = AjaxDataSource(data_url=initial_state, method='GET')

TOOLTIPS = [
    ("Year", "@Year"),
    ("Value", "@Value"),
]

#CREATE "CANVAS" FOR THE PLOT:
p = figure(plot_width=800, plot_height=300, tooltips=TOOLTIPS)
p.y_range.start = 0

#STYLE PLOT ELEMENTS
p.grid.visible = False
p.title.align='center'

#ADD GLYPHS (visual elements)
p.line('Year', 'Value', source=source)
p.circle('Year', 'Value', source=source)

#DEFINE DROPDOWNS & CALLBACKS:

select_hb = Select(title="Select Health Board:", value="NHS Tree",
                options=["NHS Tree", "NHS Rock", "NHS Train"])

select_measure = Select(title="Select measure:", value="Big Count",
                       options=["Big Count", "Small Rate"])

callback_hb = CustomJS(args=dict(source=source), code="""
    if (typeof measure === 'undefined'){
    measure = 'Big Count'; //default value
    };

    hb = cb_obj.value;
    var new_url = `http://gherka.pythonanywhere.com/data/HB=${hb}&Sex=Male&Measure=${measure}`;

    fetch(new_url).then(function(response) { return response.json(); })
    .then(function(result) {source.data = result;})
    
    source.change.emit();
""")

callback_measure = CustomJS(args=dict(source=source), code="""
    if (typeof hb === "undefined"){
    hb = 'NHS Tree'; //default value
    };

    measure = cb_obj.value;
    var new_url = `http://gherka.pythonanywhere.com/data/HB=${hb}&Sex=Male&Measure=${measure}`;

    fetch(new_url).then(function(response) { return response.json(); })
    .then(function(result) {source.data = result;})

    source.change.emit();
""")

select_measure.js_on_change('value', callback_measure)
select_hb.js_on_change('value', callback_hb)

layout = column([row([select_hb, select_measure]),p])
show(layout)